For now writing in jupyter notebook to make things clearer

In [2]:
# import necessary libraries
import pandas as pd
from itertools import product
import submitit
import os
from concurrent.futures import ProcessPoolExecutor
import numpy as np
from scipy.optimize import linear_sum_assignment
from plotnine import *
import sys
import json
import pickle
from sklearn.model_selection import train_test_split

colors_dict = {
"SCoNE":"#2f4b7c",
"MVBC":"#665191",
"RGWAS":"#a05195",  
"C-NMF":"#d45087",  
"C-CoNE":"#f95d6a",  
"G-NMF":"#ff7c43",  
"G-CoNE":"#ffa600",  
"HNMF":"#2ca02c",  
"SCoNE(Fro)":"#1f77b4",
"CoNE":"#17becf"
}

# make tmp dir
root_dir = '/gpfs/commons/groups/gursoy_lab/anewbury/unsupervised_pheno'
code_dir = f'{root_dir}/code'
tmp_folder = f"{root_dir}/output/tmp"
os.makedirs(tmp_folder, exist_ok=True)
sys.path.append(code_dir)
import evaluation.reconstruction_evaluation as reconstruction_evaluation
from simulate_data import *
from utilities import _call_kwargs_deploy_train_run
import pickle

In [3]:
variable_name = "rho"
variable_range = [0.5,1.0]
output_dir = f'{root_dir}/output/models'

In [4]:
# ---- simulate data ----
for variable in variable_range:
    sim_kwargs = {"n":500,"M_C":20,"num_genes":20,"noise":0.5,"ZU_weight": 0.5,"sparsity":0,"rho":0.8,"seed":0} 
    sim_kwargs[variable_name] = variable
    sim = simulate_views(**sim_kwargs) 
    with open(f'{tmp_folder}/sim_{variable_name}_{variable}.pkl','wb') as f:
        pickle.dump(sim,f)
    # write G,C,Z to paths
    np.save(f'{tmp_folder}/{variable_name}_{variable}_G',sim["G"])
    np.save(f'{tmp_folder}/{variable_name}_{variable}_C',sim["C"])
    np.save(f'{tmp_folder}/{variable_name}_{variable}_Z',sim["Z"])

In [5]:
# ---- config -----
lambda_options = [0,1e-6,1e-5,1e-4,1e-3,1e-2,1e-1,1,10,100] 
tuning_run_names = ['SCoNE' ,'SCoNE(Fro)','MVBC']
training_run_names = ['G-NMF','C-NMF','G-CoNE','C-CoNE','HNMF','HNMF(res)','CoNE','SCoNE','SCoNE(Fro)','RGWAS','MVBC']

In [29]:
def is_sparse(run_name): return run_name in {'SCoNE','SCoNE(Fro)','MVBC'} # runs that have sparsity params
lambda_options = [0, 1e-4, 1e-3, 1e-2, 1e-1, 1]
tuning_run_names = ['SCoNE' ,'SCoNE(Fro)','MVBC']
training_run_names = ['G-NMF','C-NMF','G-CoNE','C-CoNE','HNMF','SCoNE','SCoNE(Fro)','RGWAS','MVBC']
plan = pd.DataFrame(
    [dict(variable=variable,
        run_name=rn,
        init_name=init_name,
        split=split,
        lambda_option=(lam if is_sparse(rn) else 0))
    for variable in variable_range
    for rn in tuning_run_names
    for init_name in range(10)
    for split in ['tuning']
    for lam in (lambda_options if is_sparse(rn) else [0])]
)
new_rows = pd.DataFrame(
    [dict(variable=variable,
        run_name=rn,
        init_name=init_name,
        split=split,
        lambda_option=(lam if is_sparse(rn) else 0))
    for variable in variable_range
    for rn in training_run_names
    for init_name in range(10)
    for split in ['training']
    for lam in (lambda_options if is_sparse(rn) else [0])]
)
plan = pd.concat([plan, new_rows], ignore_index=True)

In [33]:
plan[plan['split']=='tuning'].head(20)

,variable,run_name,init_name,split,lambda_option
0,0.5,SCoNE,0,tuning,0.0000
1,0.5,SCoNE,0,tuning,0.0001
2,0.5,SCoNE,0,tuning,0.0010
3,0.5,SCoNE,0,tuning,0.0100
4,0.5,SCoNE,0,tuning,0.1000
5,0.5,SCoNE,0,tuning,1.0000
6,0.5,SCoNE,1,tuning,0.0000
7,0.5,SCoNE,1,tuning,0.0001
8,0.5,SCoNE,1,tuning,0.0010
9,0.5,SCoNE,1,tuning,0.0100


In [18]:
# deploy tuning runs
tuning_args, tuning_rows = [], []
for idx, row in plan[plan['split']=='tuning'].iterrows():
    G_path = f'{tmp_folder}/{variable_name}_{row.variable}_G.npy'
    C_path = f'{tmp_folder}/{variable_name}_{row.variable}_C.npy'
    Z_path = f'{tmp_folder}/{variable_name}_{row.variable}_Z.npy'
    G = np.load(G_path)
    C = np.load(C_path)
    Z = np.load(Z_path)
    if row.lambda_option != 0: alpha = max(G.max(), C.max())**2
    else: alpha=0
    reg = {'lambda_W':row.lambda_option,'alpha':alpha,'lambda_H_G':row.lambda_option,'lambda_H_C':row.lambda_option}
    
    a = dict(job_id=row.job_id,run_name=row.run_name, out_path=row.out_path, G=G, C=C, Z=Z, reg_params=reg,variable=row.variable,init_name=row.init_name,
        lambda_Gloss=1, 
        G_path=G_path, C_path=C_path, Z_path=Z_path, r_path='/gpfs/commons/home/anewbury/miniconda/bin/Rscript', rank=3, num_init=1, n_jobs=1) 
    tuning_args.append(a)
    tuning_rows.append(dict(out_path=row.out_path,variable=row.variable,
                            run_name=row.run_name,job_id=row.job_id,
                            lambda_option=row.lambda_option, alpha=reg['alpha'], lambda_W=reg['lambda_W'],
                            lambda_H_G=reg['lambda_H_G'],lambda_H_C=reg['lambda_H_C'], rank=3, num_init=1, 
                                lambda_Gloss=1))
tune_record = pd.DataFrame(tuning_rows)

In [19]:
from test_reconstruction import run_evaluation

In [20]:
all_results = []
computation_times_train = {}
for tune_arg in tuning_args[:20]:
    variable_val = tune_arg.pop("variable")
    init_name = tune_arg.pop("init_name")
    job_id, computation_time = _call_kwargs_deploy_train_run(tune_arg)
    computation_times_train[job_id] = computation_time
    # evaluation
    with open(f'{tmp_folder}/sim_{variable_name}_{variable_val}.pkl','rb') as f:
        sim = pickle.load(f)
    results = run_evaluation(tune_arg["run_name"],tune_arg["out_path"],sim, init_name, variable_name, variable_val, tune_arg["reg_params"]["lambda_H_G"], rank=3)
    all_results.append(results)

tune_record["computation_time"] = tune_record["job_id"].map(computation_times_train)
tune_record.to_csv(f"{output_dir}/run_record_tune.csv", index=False)
print("done with tuning", flush=True)

done with tuning


In [24]:
pd.concat(all_results)

,run_name,factor_matrix,init,sim,rho,rel_error_G,rel_error_C,lambda_val,rank
0,SCoNE,W_C,0,0.708879,0.5,0.685486,0.704969,0.0,3
1,SCoNE,W_G,0,0.789813,0.5,0.685486,0.704969,0.0,3
2,SCoNE,H_G,0,0.955693,0.5,0.685486,0.704969,0.0,3
3,SCoNE,U_G,0,0.907436,0.5,0.685486,0.704969,0.0,3
4,SCoNE,H_C,0,0.831833,0.5,0.685486,0.704969,0.0,3
...,...,...,...,...,...,...,...,...,...
1,SCoNE,W_G,1,0.640664,0.5,0.838618,0.895345,100.0,3
2,SCoNE,H_G,1,0.818551,0.5,0.838618,0.895345,100.0,3
3,SCoNE,U_G,1,0.706547,0.5,0.838618,0.895345,100.0,3
4,SCoNE,H_C,1,0.762666,0.5,0.838618,0.895345,100.0,3
